# APH/PPH Split

This notebook checks the APH/PPH split and hemorrhage logic:

1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence

In [1]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import math

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    SIMULATION_EVENT_NAMES,
    SIMULATION_STEPS,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [6]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

"""# Filter out OralIronEffectOnHemoglobin component
base_spec.components.vivarium_gates_mncnh.components = [
    c for c in base_spec.components.vivarium_gates_mncnh.components
    if "InterventionRiskEffect('misoprostol')" not in c and "OralIronEffectOnHemoglobin" not in c
]
display("InterventionRiskEffect('misoprostol')" in base_spec.components.vivarium_gates_mncnh.components)
display("OralIronEffectOnHemoglobin" in base_spec.components.vivarium_gates_mncnh.components)"""

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 60_000

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

# TODO tdy fix this to use a while loop
def run_to_step(sim: InteractiveContext, step_name: str):
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

Using artifact: /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf
Using draw column: draw_60
Population size: 60000


## 1) APH/PPH incidence and severity vs artifact

In [86]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

2026-06-23 13:49:52.994 | INFO     | simulation_20-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-23 13:49:52.996 | INFO     | simulation_20-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-23 13:49:52.998 | INFO     | simulation_20-artifact_manager:82 - Artifact additional filter terms are None.


/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/pytho

2026-06-23 13:50:22.041 | WARNING  | simulation_20-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 13:50:22.042 | WARNING  | simulation_20-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 13:50:22.160 | WARNING  | simulation_20-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-23 13:50:22.161 | WARNING  | simulation_20-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-23 13:50:22.161 | WARNING  | simulation_20-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-23 13:50:22.162 | WARNING  | simulation_20-lookup_table_manager

2026-06-23 13:50:24.347 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-23 13:50:35.278 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-23 13:50:36.114 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-23 13:50:37.300 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-23 13:50:49.323 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-23 13:51:02.858 | INFO     | simulation_20 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-23 13:51:03.414 | WARNING  | simulation_20-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-23 13:51:03.500 | INFO     | simulation_20 - vivarium.

,cause,sim_incidence,artifact_incidence,incidence_ratio_sim_over_artifact
0,antepartum_hemorrhage,0.048850,0.048107,1.015448
1,postpartum_hemorrhage,0.099419,0.099275,1.001453


## 2) Only severe hemorrhage cases die from hemorrhage

In [87]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[APH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[PPH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

2026-06-23 13:52:11.152 | INFO     | simulation_21-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-23 13:52:11.153 | INFO     | simulation_21-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-23 13:52:11.155 | INFO     | simulation_21-artifact_manager:82 - Artifact additional filter terms are None.


/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/mnt/share/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/pytho

2026-06-23 13:52:39.139 | WARNING  | simulation_21-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 13:52:39.140 | WARNING  | simulation_21-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-23 13:52:39.268 | WARNING  | simulation_21-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-23 13:52:39.269 | WARNING  | simulation_21-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-23 13:52:39.270 | WARNING  | simulation_21-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-23 13:52:39.272 | WARNING  | simulation_21-lookup_table_manager

2026-06-23 13:52:41.421 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-23 13:52:52.648 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-23 13:52:53.470 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-23 13:52:54.786 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-23 13:53:06.642 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-23 13:53:20.197 | INFO     | simulation_21 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-23 13:53:20.749 | WARNING  | simulation_21-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-23 13:53:20.833 | INFO     | simulation_21 - vivarium.

## 3) Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence, severe / total incidence are the same bt aph and pph, and the aph/pph incidence fraction matches the artifact postpartum fraction

In [3]:
art_pp_frac = artifact.load("cause.maternal_hemorrhage.postpartum_fraction").mean(axis=1).reset_index()
art_pp_frac_male = art_pp_frac.copy()
art_pp_frac_male["sex"] = "Male"
art_pp_frac = pd.concat([art_pp_frac, art_pp_frac_male]).set_index(["sex", "age_start", "age_end", "year_start", "year_end"]).rename(columns={0: "value"})
pop_structure = artifact.load("population.structure").xs("Ethiopia", level="location").rename(columns={"value": "pop_struc"})
weighted_pp_frac = pd.concat([art_pp_frac, pop_structure], axis=1).dropna()
weighted_pp_frac["pop_struc"] = weighted_pp_frac["pop_struc"] / weighted_pp_frac["pop_struc"].sum()
assert abs(weighted_pp_frac["pop_struc"].sum() - 1) < (.1 ** 6) # rounds to 1
weighted_pp_frac = (weighted_pp_frac["value"] * weighted_pp_frac["pop_struc"]).sum()
weighted_pp_frac


0.5219442318792247

In [7]:
APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

def run_sim_for_pph_summary(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)



    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.ANTEPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
        COLUMNS.PREGNANCY_OUTCOME,
        COLUMNS.MOTHER_AGE,
        APH_SEVERITY_COL,
        PPH_SEVERITY_COL,
    ])
    return pop

In [8]:
from vivarium_gates_mncnh.constants.data_values import PREGNANCY_OUTCOMES

def analyze_scenario_population(pop_table: pd.DataFrame, scenario: str, group: str):
    pop = pop_table.copy()

    # assert that for both aph and pph the set of people with moderate or severe incidence is exactly, 
    # and that theres only mod and sev for each
    assert ((pop[APH_SEVERITY_COL] == "moderate") | (pop[APH_SEVERITY_COL] == "severe")).equals(pop[COLUMNS.ANTEPARTUM_HEMORRHAGE])
    assert set(pop[pop[COLUMNS.ANTEPARTUM_HEMORRHAGE]][APH_SEVERITY_COL].tolist()) == {"moderate", "severe"}
    assert set(pop[pop[COLUMNS.POSTPARTUM_HEMORRHAGE]][PPH_SEVERITY_COL].tolist()) == {"moderate", "severe"}

    # main table row
    aph_count = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].sum()
    pph_count = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].sum()
    n_preg = len(pop)
    n_ft_preg = (pop[COLUMNS.PREGNANCY_OUTCOME] != PREGNANCY_OUTCOMES.PARTIAL_TERM_OUTCOME).sum()
    aph_sev_count = (pop[APH_SEVERITY_COL] == "severe").sum()
    pph_sev_count = (pop[PPH_SEVERITY_COL] == "severe").sum()
    main_row = {
        'scenario': scenario, 
        'group': group, 
        'aph_incidence': aph_count / n_preg, 
        'pph_incidence': pph_count / n_ft_preg, 
        'pp_frac_all_preg:': pph_count / (aph_count + pph_count),  # TODO make sure this matches loader
        'aph_sev_frac': aph_sev_count / aph_count, 
        'pph_sev_frac': pph_sev_count / pph_count,
        'n_preg': n_preg,
        'n_ft_preg': n_ft_preg
    }

    def get_sev_frac_rows_by_strat(curpop, strat_col_name):
        sev_frac_strat_rows = []
        strat_vals = curpop[strat_col_name].unique()
        for strat_val in strat_vals:
            for timing in ['aph', 'pph']:
                timing_col_name = COLUMNS.ANTEPARTUM_HEMORRHAGE if timing == 'aph' else COLUMNS.POSTPARTUM_HEMORRHAGE
                sev_col_name = APH_SEVERITY_COL if timing == 'aph' else PPH_SEVERITY_COL
                pop_strat = curpop[(curpop[strat_col_name] == strat_val) & (curpop[timing_col_name])]
                sev_count = (pop_strat[sev_col_name] == "severe").sum()
                n = len(pop_strat)
                sev_frac_strat_rows.append({
                    'scenario': scenario,
                    'group': group,
                    'timing': timing,
                    strat_col_name: strat_val,
                    'val': math.nan if n==0 else sev_count / n,
                    'n': n
                })
        return sev_frac_strat_rows
    
    # check severity fraction for aph and pph by age groups, delivery location and pregnancy outcome
    sev_frac_rows = []
    sev_frac_rows.extend(get_sev_frac_rows_by_strat(pop, COLUMNS.MOTHER_AGE))
    sev_frac_rows.extend(get_sev_frac_rows_by_strat(pop, COLUMNS.DELIVERY_FACILITY_TYPE))
    sev_frac_rows.extend(get_sev_frac_rows_by_strat(pop, COLUMNS.PREGNANCY_OUTCOME))

    return main_row, sev_frac_rows


def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    pop = run_sim_for_pph_summary(scenario)
    pop[COLUMNS.MOTHER_AGE] = (pop[COLUMNS.MOTHER_AGE] // 5) * 5  # get age_start (ie group) from continuous age

    main_out = []
    sev_frac_out = []

    # overall group
    main_row, sev_frac_rows = analyze_scenario_population(pop, scenario, 'overall')
    main_out.append(main_row)
    sev_frac_out.extend(sev_frac_rows)

    # misoprostol=True and False group
    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) > 0:
            main_row, sev_frac_rows = analyze_scenario_population(sub, scenario, f'misoprostol_available={value}')
            main_out.append(main_row)
            sev_frac_out.extend(sev_frac_rows)

    # at-home group
    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    main_row, sev_frac_rows = analyze_scenario_population(home, scenario, 'home_only')
    main_out.append(main_row)
    sev_frac_out.extend(sev_frac_rows)

    return pd.DataFrame(main_out), pd.DataFrame(sev_frac_out)

baseline_main, baseline_sev_frac = pph_summary_for_scenario('baseline')
miso_vv_main, miso_vv_sev_frac = pph_summary_for_scenario('misoprostol_vv')

res_main = pd.concat([baseline_main, miso_vv_main], ignore_index=True)
res_sev_frac = pd.concat([baseline_sev_frac, miso_vv_sev_frac], ignore_index=True)

2026-06-24 13:24:01.963 | INFO     | simulation_2-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-06-24 13:24:02.059 | INFO     | simulation_2-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-06-24 13:24:02.060 | INFO     | simulation_2-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-06-24 13:25:14.065 | WARNING  | simulation_2-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-24 13:25:14.066 | WARNING  | simulation_2-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-24 13:25:14.566 | WARNING  | simulation_2-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-24 13:25:14.567 | WARNING  | simulation_2-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-24 13:25:14.567 | WARNING  | simulation_2-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-24 13:25:14.568 | WARNING  | simulation_2-lookup_table_manager:85 - 

2026-06-24 13:25:20.369 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-24 13:25:41.904 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-24 13:25:43.028 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-24 13:25:44.524 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-24 13:26:08.629 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-24 13:26:35.011 | INFO     | simulation_2 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-24 13:26:35.729 | WARNING  | simulation_2-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-24 13:26:35.863 | INFO     | simulation_2 - vivarium.framewor

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-06-24 13:28:21.890 | WARNING  | simulation_3-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-24 13:28:21.891 | WARNING  | simulation_3-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-06-24 13:28:22.197 | WARNING  | simulation_3-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-06-24 13:28:22.198 | WARNING  | simulation_3-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-06-24 13:28:22.199 | WARNING  | simulation_3-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-06-24 13:28:22.200 | WARNING  | simulation_3-lookup_table_manager:85 - 

2026-06-24 13:28:24.977 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-06-24 13:28:45.342 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-06-24 13:28:46.442 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-06-24 13:28:47.932 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-06-24 13:29:11.974 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-06-24 13:29:38.434 | INFO     | simulation_3 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-06-24 13:29:39.145 | WARNING  | simulation_3-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-06-24 13:29:39.260 | INFO     | simulation_3 - vivarium.framewor

In [9]:
res_main

,scenario,group,aph_incidence,pph_incidence,pp_frac_all_preg:,aph_sev_frac,pph_sev_frac,n_preg,n_ft_preg
0,baseline,overall,0.048850,0.099419,0.534836,0.155578,0.163501,60000,33897
1,baseline,misoprostol_available=False,0.048850,0.099419,0.534836,0.155578,0.163501,60000,33897
2,baseline,home_only,0.050783,0.102035,0.667692,0.158730,0.177749,14887,14887
3,misoprostol_vv,overall,0.048850,0.095377,0.524497,0.155578,0.163006,60000,33897
4,misoprostol_vv,misoprostol_available=True,0.045499,0.064964,0.588106,0.133690,0.183521,4110,4110
5,misoprostol_vv,misoprostol_available=False,0.049096,0.099574,0.519440,0.157070,0.161160,55890,29787
6,misoprostol_vv,home_only,0.050783,0.092833,0.646399,0.158730,0.178003,14887,14887


In [14]:
res_sev_frac[(res_sev_frac.scenario == "baseline") & (res_sev_frac.group == "home_only") & (res_sev_frac.timing == "pph") ]

,scenario,group,timing,age,val,n,delivery_facility_type,pregnancy_outcome
65,baseline,home_only,pph,25.0,0.178161,174,NaN,NaN
67,baseline,home_only,pph,15.0,0.126214,103,NaN,NaN
69,baseline,home_only,pph,40.0,0.197080,137,NaN,NaN
71,baseline,home_only,pph,30.0,0.157068,382,NaN,NaN
73,baseline,home_only,pph,20.0,0.195011,441,NaN,NaN
75,baseline,home_only,pph,35.0,0.180723,249,NaN,NaN
77,baseline,home_only,pph,45.0,0.277778,18,NaN,NaN
79,baseline,home_only,pph,10.0,0.200000,15,NaN,NaN
81,baseline,home_only,pph,50.0,NaN,0,NaN,NaN
83,baseline,home_only,pph,NaN,0.177749,1519,home,NaN


use aph/pph incidence rate keys as targets instead of pp fraction, only subset for pph
if dont subset to partial term, should match pp_frac, but


Severities
Overlapping azithro coverage? Presence of someting that reduceds a diff maternal disorder than hemorrhage for different delivery location
Severity should be very simple, nothing should modify?
Age? severity is age specifc, but barely
Aph mortality should be leaving behind less risky people
any group of ppl who have aph should have same severity percentage....
**checks of things we think should be true mentioned above**
should have table of aph/pph severity value counts grouped by different things, asserts as above
TODO only run sim once

In [84]:
art_sev_frac = artifact.load("cause.maternal_hemorrhage.severe_fraction").mean(axis=1) # barely varies by age
art_sev_frac = art_sev_frac[art_sev_frac != 0].mean()
targets = pd.concat([check4["weighted_pp_frac"], check4["art_sev_frac"], check4["art_sev_frac"]], axis=1)
targets.columns = ["pph_incidence_frac", "aph_sev_frac", "pph_sev_frac"]
check4 = check4.drop(columns=["weighted_pp_frac", "art_sev_frac"])
targets

KeyError: 'weighted_pp_frac'

In [ ]:
# number of standard errors away from targets
n_std_errors = pd.concat([check4["n"] * (check4["aph_incidence"] + check4["pph_incidence"]), check4["n"] * check4["aph_incidence"], check4["n"] * check4["pph_incidence"]], axis=1)
n_std_errors = n_std_errors.rename(columns={0: "pph_incidence_frac", 1: "aph_sev_frac", 2: "pph_sev_frac"})
check4copy = check4.reset_index().copy().drop(columns=["aph_incidence", "pph_incidence", "aph_incidence_frac", "n"]).set_index(["scenario", "group"])
n_std_errors.index = check4copy.index
std_error = np.sqrt(check4copy * (1 - check4copy) / n_std_errors)
std_error
(check4copy - targets) / std_error

pph_incidence_frac  aph_sev_frac  pph_sev_frac
scenario       group                                                                      
baseline       overall                                2.051611      0.927894      2.218854
               misoprostol_available=False            2.051611      0.927894      2.218854
               home_only                             14.758260      0.704575      2.893496
misoprostol_vv overall                                0.401336      0.927894      2.099742
               misoprostol_available=True             2.864261     -0.629909      1.441745
               misoprostol_available=False           -0.378812      1.109068      1.746899
               home_only                             12.036672      0.704575      2.783105

In [49]:
artifact.load("cause.maternal_hemorrhage.incidence_rate").mean(axis=1).reset_index()

,sex,age_start,age_end,year_start,year_end,0
0,Female,0.000000,0.019178,2023,2024,0.000000
1,Female,0.019178,0.076712,2023,2024,0.000000
2,Female,0.076712,0.500000,2023,2024,0.000000
3,Female,0.500000,1.000000,2023,2024,0.000000
4,Female,1.000000,2.000000,2023,2024,0.000000
5,Female,2.000000,5.000000,2023,2024,0.000000
6,Female,5.000000,10.000000,2023,2024,0.000000
7,Female,10.000000,15.000000,2023,2024,0.000911
8,Female,15.000000,20.000000,2023,2024,0.012072
9,Female,20.000000,25.000000,2023,2024,0.039736


Home is higher. Misoprostol should be lowering pph, this seems backwards. 
Within home, misoprostol_available=True is subset, and pph is lower which is the effect we want. Effect seems going right way.
But home has much higher pph 
confounding path - anc delivery mechanism for ifa. so ppl at home more likely to not have ifa, and have lower hemoglobin. but lower hemoglobin should be increasing risk of aph and pph equally. 
does this table include a/e/m in pop?
IFD ANC correlation like we were looking at last time?
-Could my pp_Frac be wrong for this pop?

drill in on one number that seems weird, how does it correspond to other factors. checking misoprostol true is actually subset of home, what else does it correlate with

2nd order =Misoprostol=true only happens when you survive pregnancy. aph_sev_frac low bc ppl who died from aph are not able to get misoprostol





In [ ]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])